# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

- The `mlcroissant.Dataset` object loads the Croissant schema and enables exploration of its structure and data.
- The dataset metadata is displayed for context.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset Croissant object
dataset = mlc.Dataset(url)

# Display main metadata
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")
print(f"Keywords: {', '.join(dataset.metadata.keywords or [])}")
print(f"Cite as: {dataset.metadata.citeAs}")

## 2. Data Overview
Review available record sets, fields, and their IDs (all referenced by `@id`).

- **RecordSet**: Describes a logical table of records, the main entry to the dataset. Each usually corresponds to a file or table.
- **Field**: Individual columns/features within a record set.

Let's examine all available record sets and their IDs (using the Croissant schema `@id`).

In [ ]:
# List available RecordSets and their field @ids
print("Available RecordSets in this dataset:")
record_set_ids = []
for rset in dataset.record_sets:
    print(f"- RecordSet name: {rset.name}")
    print(f"  @id: {rset.id}")
    print(f"  Description: {rset.description}")
    print(f"  Fields in this RecordSet:")
    for field in rset.fields:
        print(f"    - Field name: {field.name}, @id: {field.id}, dataType: {field.data_type}")
    print()
    record_set_ids.append(rset.id)
if not record_set_ids:
    print("No record sets found in the Croissant schema (check the dataset schema or contact the provider).")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the previous overview.

- If the dataset has multiple record sets, loop through each; otherwise, select one for demonstration.
- Note: All references to entities are made using their `@id`.

**Example uses first available record set (if found).**

In [ ]:
# Extract data from each record set
if not record_set_ids:
    print("No record sets available in the dataset; cannot extract records.")
    dataframes = {}
else:
    dataframes = {}
    for record_set_id in record_set_ids:
        print(f"Loading records for RecordSet: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for {record_set_id}\nColumns: {df.columns.tolist()}")
        else:
            print(f"No records found for {record_set_id}.")

    # For demonstration, pick the first one with data
    main_record_set_id = None
    for rsid in record_set_ids:
        if rsid in dataframes and not dataframes[rsid].empty:
            main_record_set_id = rsid
            break
    if main_record_set_id:
        print(f"\nUsing record set {main_record_set_id} for further steps.\n")
        print("Data sample:")
        display(dataframes[main_record_set_id].head())
    else:
        print("No non-empty record sets found for further analysis.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

- Choose a numeric field by its `@id` from the record set (see previous cells for available fields and their data types).
- Here, we'll attempt to select a numeric column if one is present, otherwise instruct the user.

In [ ]:
import numpy as np

# Proceed only if a main record set is available
if not record_set_ids or not dataframes:
    print("No data loaded for EDA.")
else:
    df = dataframes[main_record_set_id]
    # Try to auto-detect a numeric field
    # We'll pick a column that is float or int and has unique values > 5
    numeric_field = None
    for col in df.columns.tolist():
        if np.issubdtype(df[col].dtype, np.number) and df[col].nunique() > 5:
            numeric_field = col
            break

    if not numeric_field:
        print("No suitable numeric field found for EDA in the current RecordSet.")
    else:
        print(f"Using numeric field for EDA: {numeric_field}")

        threshold = df[numeric_field].mean() if np.issubdtype(df[numeric_field].dtype, np.number) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean)")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by another field if a non-numeric, non-unique field is available
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].dtype == object and df[col].nunique() > 1 and df[col].nunique() < len(df)//2:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped data by {group_field} (mean of {numeric_field}):")
            display(grouped_df.head())
        else:
            print("No suitable group field found for aggregation.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

- We'll display a histogram of the numeric field and, if a group field was detected, boxplots by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not record_set_ids or not dataframes:
    print("No data loaded for visualization.")
elif not numeric_field or numeric_field not in filtered_df:
    print("No suitable numeric field found for visualization.")
else:
    plt.figure(figsize=(8,5))
    sns.histplot(filtered_df[numeric_field], kde=True)
    plt.title(f'Histogram of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    if 'group_field' in locals() and group_field and group_field in filtered_df:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
        plt.title(f'{numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to explore the Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells dataset. We loaded the Croissant schema, reviewed metadata and record sets (referencing all entities by their `@id`), extracted records into DataFrames, applied exploratory data analysis methods including normalization and grouping, and visualized key distributions.

- **Key Findings**: (To be completed after running analysis; summarize any trends, outliers, group differences, etc. discovered.)

For further analysis, refer to the dataset's Croissant documentation and field descriptions for domain-specific insights.